# Embedding実験ノート

このノートブックでは、OpenAI Embeddingの基本的な使い方と類似度計算を学びます。

## 学習目標
1. OpenAI Embedding APIの基本操作
2. コサイン類似度の計算と解釈
3. 記事データを使った類似検索の実験

## 1. 環境設定

In [ ]:
# 必要なライブラリのインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

# .envファイルから環境変数を読み込む
load_dotenv()

# OpenAIクライアントの初期化
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete!")

## 2. Embedding生成の基本

In [ ]:
def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """テキストのEmbeddingを生成する
    
    Args:
        text: 埋め込むテキスト
        model: 使用するモデル（text-embedding-3-smallがコスト効率良い）
    
    Returns:
        1536次元のベクトル
    """
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# テスト
test_embedding = get_embedding("Hello, world!")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

## 3. コサイン類似度の計算

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """2つのベクトル間のコサイン類似度を計算する
    
    コサイン類似度 = (A・B) / (|A| * |B|)
    
    値の解釈:
    - 1.0: 完全に同じ方向（最も類似）
    - 0.0: 直交（無関係）
    - -1.0: 完全に反対の方向（最も非類似）
    
    Args:
        a: ベクトル1
        b: ベクトル2
    
    Returns:
        類似度（-1.0〜1.0）
    """
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# テスト
vec_a = get_embedding("Python is a programming language")
vec_b = get_embedding("JavaScript is a programming language")
vec_c = get_embedding("The weather is nice today")

print(f"Python vs JavaScript: {cosine_similarity(vec_a, vec_b):.4f}")
print(f"Python vs Weather: {cosine_similarity(vec_a, vec_c):.4f}")
print(f"JavaScript vs Weather: {cosine_similarity(vec_b, vec_c):.4f}")

### 考察
- PythonとJavaScriptは両方プログラミング言語なので、類似度が高い（0.8以上になるはず）
- プログラミングと天気は関係ないので、類似度が低い（0.5以下になるはず）

**思考実験**: なぜコサイン類似度がEmbeddingの比較に適しているのでしょうか？
- ベクトルの大きさ（長さ）ではなく、方向だけを比較するため
- 文章の長さに影響されにくい

## 4. 記事データを使った実験

実際の技術記事のタイトルを使って類似検索を実験してみましょう。

In [ ]:
# サンプル記事タイトル（技術記事風）
sample_articles = [
    "Rustで高速なWebサーバーを構築する方法",
    "Go言語によるマイクロサービスアーキテクチャ入門",
    "Pythonで機械学習モデルをデプロイする",
    "TypeScriptの型システムを深く理解する",
    "Kubernetesでのコンテナオーケストレーション",
    "PostgreSQLのパフォーマンスチューニング",
    "React Hooksのベストプラクティス",
    "GitHub Actionsで CI/CD パイプラインを構築",
]

# 各記事のEmbeddingを生成
article_embeddings = {}
for article in sample_articles:
    article_embeddings[article] = get_embedding(article)
    print(f"Embedded: {article[:30]}...")

print(f"\nTotal: {len(article_embeddings)} articles embedded")

In [ ]:
def search_similar(query: str, embeddings: dict, top_k: int = 3) -> list[tuple[str, float]]:
    """クエリに類似した記事を検索する
    
    Args:
        query: 検索クエリ
        embeddings: 記事タイトル -> Embedding のマッピング
        top_k: 返す結果の数
    
    Returns:
        (タイトル, 類似度) のリスト（類似度降順）
    """
    query_embedding = get_embedding(query)
    
    similarities = []
    for title, emb in embeddings.items():
        sim = cosine_similarity(query_embedding, emb)
        similarities.append((title, sim))
    
    # 類似度でソート（降順）
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

# 検索テスト
queries = [
    "バックエンド開発",
    "フロントエンドフレームワーク",
    "データベース",
    "DevOps",
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    results = search_similar(query, article_embeddings)
    for title, score in results:
        print(f"   {score:.4f} - {title}")

## 5. 類似度マトリックスの可視化

In [ ]:
# 記事間の類似度マトリックスを計算
titles = list(article_embeddings.keys())
n = len(titles)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity_matrix[i, j] = cosine_similarity(
            article_embeddings[titles[i]],
            article_embeddings[titles[j]]
        )

# 短いタイトル名を作成
short_titles = [t[:15] + "..." for t in titles]

# テキストベースの表示
print("\n📊 Similarity Matrix (text display)\n")
print(" " * 18, end="")
for i, t in enumerate(short_titles):
    print(f"[{i}]", end=" ")
print()

for i, row in enumerate(similarity_matrix):
    print(f"[{i}] {short_titles[i]:15}", end=" ")
    for val in row:
        print(f"{val:.2f}", end=" ")
    print()

## 6. 次のステップ

このノートブックで学んだこと:
1. ✅ OpenAI Embedding APIの基本的な使い方
2. ✅ コサイン類似度の計算と解釈
3. ✅ 類似記事の検索

次に学ぶこと:
- pgvectorを使ったベクトル検索の効率化
- 大量のデータに対するバッチ処理
- RAGパイプラインの構築

In [ ]:
# catchup-aiのモジュールを使った例（実装後）
# from catchup_ai.core.embedding import OpenAIEmbeddingAdapter, ArticleEmbeddingInput
# 
# adapter = OpenAIEmbeddingAdapter()
# article = ArticleEmbeddingInput(
#     article_id=1,
#     title="Rustで高速なWebサーバーを構築する方法",
#     content="Rustは..."
# )
# result = adapter.embed_article(article)
# print(f"Dimension: {result.dimension}")